# Convolution of Periodic Splines
The data samples at the integers are represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line; the domain of the curves encompasses two periods. The thin blue curve is the result of the operation applied to the thick gray curve.

In [113]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic linear spline with normal Gaussian coefficients
s1 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 1)
# Initial random periodic cubic spline with absolute Cauchy coefficients
s2 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_cauchy(6), degree = 3)
s2.spline_coeff = np.abs(s2.spline_coeff)
s2 = s2.times(1.0 / np.sum(s2.spline_coeff))

# Plot
def update_plot (
    period = 6,
    sep1 = "",
    degree1 = 1,
    delay1 = 0.0,
    sep2 = "",
    degree2 = 3,
    delay2 = 0.0
):
    global s1
    global s2

    # Update of the Gaussian spline
    if s1.period != period:
        s1 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = s1.degree
        )
    s1.degree = degree1
    s1.delay = delay1
    # Update of the Cauchy spline
    if s2.period != period:
        s2 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_cauchy(period),
            degree = s2.degree
        )
        s2.spline_coeff = np.abs(s2.spline_coeff)
        s2 = s2.times(1.0 / np.sum(s2.spline_coeff))
    s2.degree = degree2
    s2.delay = delay2

    # Convolution
    s3 = sk.PeriodicSpline1D.convolve(s1, s2)

    # Dynamic range
    image = {s1.image(), s2.image(), s3.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plot of the splines
    subplot = plt.subplots()
    s1.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "#E0E0E0",
        curve_lw = 3.0,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " "
    )
    s2.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "-C5",
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " "
    )
    s3.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        knot_marker = " "
    )
    plt.show()

# Interaction
widgets.interactive(
    update_plot,
    period = (1, max_period),
    sep1 = widgets.HTML(
        value="<hr style='border:1px solid;margin:15px 0;width:135px'>",
        description = "• • • • •"
    ),
    degree1 = (0, max_degree),
    delay1 = (-max_delay, max_delay),
    sep2 = widgets.HTML(
        value="<hr style='border:1px solid;margin:15px 0;width:135px'>",
        description = "• • • • •"
    ),
    degree2 = (0, max_degree),
    delay2 = (-max_delay, max_delay)
)


interactive(children=(IntSlider(value=6, description='period', max=15, min=1), HTML(value="<hr style='border:1…